# Gold ao vivo — `gold_orders_live`

Mantém o **estado atual de cada pedido** a partir do feed de eventos de status da bronze, tolerando eventos duplicados e fora de ordem. Desenho: `docs/design/fase-5-streaming.md`; decisões: ADRs 0009 a 0012.

| Peça | Decisão |
|---|---|
| Execução | Structured Streaming com `foreachBatch`, gatilho `AvailableNow` (o serverless não suporta `ProcessingTime`) |
| Deduplicação | `dropDuplicatesWithinWatermark` sobre `(order_id, event_sequence)`, watermark de 4 dias |
| Escrita | `MERGE` que só troca o status se `event_sequence` for **maior** que a gravada (idempotente) |
| Tabela | `olist.gold.gold_orders_live`, uma linha por pedido (tabela Delta própria, fora do pipeline `olist.gold`) |

**Parâmetro**

| Widget | Padrão | Uso |
|---|---|---|
| `run_id` | vazio | identificador da execução; no job recebe `{{job.run_id}}`, vazio gera um UUID |
| `teste_leitura` | `nao` | `leitura` roda o teste da leitura em streaming; `robustez` testa mudanças na tabela de origem (cópia); ambos encerram o notebook (etapa 5.2) |

**Partes** (o arquivo é entregue completo, mas cresce a cada parte)

1. Configuração, tabelas, leitura em streaming, redução do micro-lote e **teste da leitura em streaming** (etapas 5.1 e 5.2)
2. `foreachBatch` + `MERGE`, `ops.streaming_batches` e execução (etapa 5.3)
3. Regra de risco e lista de pedidos em risco (etapa 5.4)
4. Validação e idempotência (etapa 5.4)

In [ ]:
# O QUE FAZ: cria o parâmetro do notebook (caixa de texto no topo da página).
#
# FUNÇÃO:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto; o valor é lido
#   depois com dbutils.widgets.get(nome) e sempre chega como string.
# - dbutils.widgets.dropdown(nome, valor_padrão, lista, rótulo): widget de lista suspensa (teste_leitura:
#   nao, leitura ou robustez; os testes da etapa 5.2 só rodam quando o valor não é "nao").
dbutils.widgets.text("run_id", "", "ID da execução")
dbutils.widgets.dropdown("teste_leitura", "nao", ["nao", "leitura", "robustez"], "Teste da leitura (etapa 5.2)")

In [ ]:
# O QUE FAZ: define as constantes do notebook: tabelas de origem e de destino, pasta do checkpoint,
#   tamanho do watermark, SLA e a configuração da regra de risco (ADR 0010, 0011 e 0012).
#   Todos os nomes derivam de CATALOG. Nada aqui lê ou grava dados.
#
# RECURSOS:
# - f"...{variavel}...": f-string, monta texto substituindo as variáveis entre chaves.
# - WATERMARK_DELAY: tempo (em tempo do EVENTO, não do relógio) que o Spark espera por dados
#   atrasados antes de esquecer um evento já visto. O simulador atrasa até 3 dias; 4 dá margem.
# - Checkpoint: pasta onde o Spark guarda até onde já leu e o estado da deduplicação. Apagar a
#   pasta faz o stream recomeçar do zero (o MERGE idempotente torna isso seguro, só mais caro).
# - /Volumes/<catálogo>/<schema>/<volume>: caminho de um Volume do Unity Catalog.
# - uuid.uuid4(): gera um identificador aleatório único; str() converte em texto.
# - "a or b": devolve "a" se tiver valor (texto não vazio); senão, "b".
# - str.strip(): remove espaços e quebras de linha do início e do fim do texto.
import uuid

CATALOG = "olist"

EVENTS_TABLE = f"{CATALOG}.bronze.order_status_events"
ORDERS_TABLE = f"{CATALOG}.bronze.orders"          # data estimada de entrega (etapa 5.3)
LIVE_TABLE = f"{CATALOG}.gold.gold_orders_live"
BATCHES_TABLE = f"{CATALOG}.ops.streaming_batches"  # um registro por micro-lote (etapa 5.3)

CHECKPOINT_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/streaming/gold_orders_live"

WATERMARK_DELAY = "4 days"   # ADR 0010
SLA_MINUTES = 15             # ADR 0012

# Regra de risco (ADR 0011): ajustar aqui, não no código da regra. Usada na etapa 5.4.
RISK_CONFIG = {
    "em_risco_dias": 2,
    "status_abertos": ["created", "approved", "invoiced", "processing", "shipped"],
}

RUN_ID = dbutils.widgets.get("run_id").strip() or str(uuid.uuid4())

print(f"run_id:     {RUN_ID}")
print(f"origem:     {EVENTS_TABLE}")
print(f"destino:    {LIVE_TABLE}")
print(f"checkpoint: {CHECKPOINT_DIR}")
print(f"watermark:  {WATERMARK_DELAY} | SLA: {SLA_MINUTES} min")

## Tabelas

`gold_orders_live` guarda **uma linha por pedido**. As colunas de etapa (`created_at` a `delivered_at`) são preenchidas por **qualquer** evento, na ordem em que chegar; já o `status`, a `event_sequence` e o `event_ts` só avançam quando chega um evento com sequência maior. Por isso um `delivered` que chega antes do `shipped` não perde o instante do despacho quando o `shipped` aparecer depois.

`ops.streaming_batches` registra cada micro-lote (volume, eventos ignorados e latência), base dos critérios de aceite de latência e de eventos tardios.

In [ ]:
# O QUE FAZ: cria as duas tabelas se ainda não existirem. Pode ser executada várias vezes: não
#   altera uma tabela que já existe.
#
# RECURSOS SQL:
# - CREATE TABLE IF NOT EXISTS: cria só se a tabela não existir.
# - COMMENT '...': descrição da tabela ou da coluna, que aparece no Catalog Explorer.
# - TIMESTAMP: data/hora com fuso da sessão; BIGINT: inteiro de 64 bits; DOUBLE: decimal.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LIVE_TABLE} (
        order_id                 STRING    COMMENT 'Identificador do pedido (chave: uma linha por pedido).',
        status                   STRING    COMMENT 'Status atual: o do evento de maior event_sequence já aplicado.',
        event_sequence           INT       COMMENT 'Sequência do último evento aplicado; só aumenta.',
        event_ts                 TIMESTAMP COMMENT 'Instante (simulado) do último evento aplicado.',
        created_at               TIMESTAMP COMMENT 'Instante do evento created (compra).',
        approved_at              TIMESTAMP COMMENT 'Instante do evento approved.',
        shipped_at               TIMESTAMP COMMENT 'Instante do evento shipped (despacho).',
        delivered_at             TIMESTAMP COMMENT 'Instante do evento delivered (entrega ao cliente).',
        estimated_delivery_date  TIMESTAMP COMMENT 'Data estimada de entrega informada na compra.',
        risco_atraso             STRING    COMMENT 'normal, em_risco ou atrasado; só pedidos abertos (ADR 0011).',
        _updated_at              TIMESTAMP COMMENT 'Momento em que a linha foi gravada pela última vez.',
        _run_id                  STRING    COMMENT 'Execução que gravou a linha pela última vez.'
    )
    COMMENT 'Estado atual de cada pedido, mantido por streaming (MERGE por event_sequence). Visão operacional ao vivo: os KPIs oficiais continuam na gold batch (ADR 0012).'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BATCHES_TABLE} (
        run_id               STRING    COMMENT 'Execução do notebook (job.run_id ou UUID).',
        batch_id             BIGINT    COMMENT 'Número do micro-lote dentro do checkpoint.',
        processed_at         TIMESTAMP COMMENT 'Momento em que o micro-lote foi gravado.',
        events_in            BIGINT    COMMENT 'Eventos que chegaram ao micro-lote depois da deduplicação.',
        orders_touched       BIGINT    COMMENT 'Pedidos distintos no micro-lote.',
        status_updated       BIGINT    COMMENT 'Pedidos cujo status avançou (sequência maior).',
        ignored_old_sequence BIGINT    COMMENT 'Pedidos cujo evento tinha sequência menor ou igual à gravada.',
        late_events          BIGINT    COMMENT 'Eventos com event_ts mais antigo que o watermark (aplicados, nunca descartados).',
        max_event_ts         TIMESTAMP COMMENT 'Maior event_ts do micro-lote (data simulada corrente).',
        max_latency_seconds  DOUBLE    COMMENT 'Maior intervalo entre a chegada do arquivo na landing e a gravação do micro-lote.'
    )
    COMMENT 'Um registro por micro-lote do streaming gold_orders_live: volume, eventos ignorados, tardios e latência (critérios 5 e 7 da Fase 5).'
""")

print("Tabelas prontas:", LIVE_TABLE, "e", BATCHES_TABLE)

## Leitura em streaming e deduplicação

A origem é a **tabela bronze**, lida como stream: a bronze é Delta *append-only*, então cada linha nova (inclusive duplicatas e reenvios de arquivo) aparece como linha nova para o stream. A deduplicação usa o tempo do evento (`event_ts`) como relógio do watermark.

A leitura tem duas camadas (`read_events_clean`, sem deduplicação, e `read_events_stream`, deduplicada) e ignora commits que reescrevem a bronze (`skipChangeCommits`), para que uma exclusão de cliente não derrube o stream. O teste da etapa 5.2, mais abaixo, mede as duas.

Pontos de atenção (a verificar na etapa 5.3):

- `dropDuplicatesWithinWatermark` lembra cada chave `(order_id, event_sequence)` só até o watermark passar. Um duplicado que chega **depois** disso não é mais reconhecido pela deduplicação; quem o impede de corromper o estado é o `MERGE` por sequência (segunda defesa, ADR 0010).
- É preciso **medir** se o Spark descarta ou deixa passar eventos mais antigos que o watermark. Se descartar, o critério 7 (evento tardio contado, nunca em silêncio) exige contá-los **antes** da deduplicação. O teste está na Parte 2.

In [ ]:
# O QUE FAZ: declara a leitura do feed de eventos da bronze em três camadas pequenas: limpeza
#   (função pura), leitura em streaming sem deduplicação e leitura em streaming deduplicada. A
#   separação deixa medir, na etapa 5.2, quantas linhas a bronze entrega e quantas sobram depois da
#   deduplicação.
#
# FUNÇÕES DECLARADAS:
# - clean_events(df): função pura (DataFrame -> DataFrame, serve para stream e para tabela): descarta a
#   linha corrompida e o pedido sem id, tipa event_sequence (int) e event_ts (timestamp) e mantém as
#   colunas de auditoria da bronze.
# - read_events_clean(table, skip_change_commits): lê a tabela como stream e aplica clean_events. Com
#   skip_change_commits = True (padrão), o stream ignora commits que alteram ou apagam linhas já
#   existentes (UPDATE, DELETE, MERGE) e só lê as linhas acrescentadas. Sem isso, qualquer DELETE na
#   bronze (como a exclusão de cliente da Fase 6) derruba o stream.
# - read_events_stream(): read_events_clean + watermark + deduplicação por (order_id, event_sequence).
#   Não executa nada: um stream só começa a rodar quando é gravado com writeStream.start().
#
# FUNÇÕES NATIVAS:
# - spark.readStream.table(tabela): lê uma tabela Delta como fonte de streaming; cada linha
#   acrescentada à tabela vira linha nova no stream.
# - .option("skipChangeCommits", "true"): opção da fonte Delta que ignora commits que reescrevem
#   dados existentes.
# - df.where(condição): filtra linhas. Column.isNull() / isNotNull(): verifica nulo.
# - "&" entre colunas: E lógico do Spark (cada lado entre parênteses, por precedência).
# - Column.cast("int"): converte o texto da bronze em inteiro.
# - F.to_timestamp(coluna, formato): texto -> TIMESTAMP. Usamos TIMESTAMP (e não TIMESTAMP_NTZ)
#   porque é o tipo de relógio mais usado pelo watermark.
# - df.withWatermark(coluna, atraso): declara a coluna de tempo do evento e quanto atraso tolerar;
#   o Spark descarta o estado de deduplicação mais antigo que (maior event_ts visto - atraso).
# - df.dropDuplicatesWithinWatermark(colunas): mantém uma linha por combinação das colunas dentro
#   do watermark. Requer o withWatermark antes.
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"


def clean_events(df: DataFrame) -> DataFrame:
    """Descarta lixo (linha corrompida, pedido sem id) e tipa as colunas. Vale para stream e para tabela."""
    return (
        df
        .where(F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull())
        .select(
            "order_id",
            "status",
            F.col("event_sequence").cast("int").alias("event_sequence"),
            F.to_timestamp(F.col("event_ts"), TS_FORMAT).alias("event_ts"),
            "_ingested_at",
            "_source_file_modified_at",
        )
    )


def read_events_clean(table: str = EVENTS_TABLE, skip_change_commits: bool = True) -> DataFrame:
    """Eventos de status da bronze como stream, limpos e tipados (sem deduplicação)."""
    reader = spark.readStream
    if skip_change_commits:
        reader = reader.option("skipChangeCommits", "true")
    return clean_events(reader.table(table))


def read_events_stream() -> DataFrame:
    """Eventos de status da bronze, tipados e deduplicados dentro do watermark (stream)."""
    return (
        read_events_clean()
        .withWatermark("event_ts", WATERMARK_DELAY)
        .dropDuplicatesWithinWatermark(["order_id", "event_sequence"])
    )

In [ ]:
# O QUE FAZ: confere que a leitura é de fato um stream e mostra o schema. Não grava nada.
#
# RECURSOS:
# - df.isStreaming: True se o DataFrame é uma consulta em streaming.
# - df.printSchema(): lista as colunas e os tipos.
preview = read_events_stream()

print(f"isStreaming: {preview.isStreaming}")
preview.printSchema()

## Redução do micro-lote

Um micro-lote pode trazer **vários eventos do mesmo pedido**, em qualquer ordem. Antes do `MERGE`, o lote é reduzido a **uma linha por pedido**: o evento de maior sequência (status atual) e o instante de cada etapa. É uma função pura (DataFrame → DataFrame), testável sem stream.

In [ ]:
# O QUE FAZ: declara a função pura que reduz um micro-lote a uma linha por pedido, e o mapa de
#   status para a coluna de instante da etapa.
#
# FUNÇÃO DECLARADA:
# - reduce_batch(events): devolve uma linha por order_id com: status, event_sequence e event_ts
#   do evento de MAIOR sequência (desempate pelo event_ts mais recente) e created_at, approved_at,
#   shipped_at e delivered_at (o instante de cada etapa que aparece no lote, nulo se não apareceu).
#
# FUNÇÕES NATIVAS:
# - Window.partitionBy(chave).orderBy(...): janela que separa as linhas por pedido e as ordena.
#   F.col(...).desc(): ordem decrescente.
# - F.row_number().over(janela): numera as linhas de cada pedido a partir de 1; a linha 1 é o evento
#   de maior sequência.
# - df.groupBy(chave).agg(...): agrega por pedido.
# - F.max(F.when(condição, valor)): maior valor entre as linhas que atendem à condição (when sem
#   otherwise devolve nulo nas demais, e max ignora nulos). Dá o instante de cada etapa.
# - df.join(outro, on=coluna, how="inner"): junta pelas colunas indicadas.
# - [... for ... in dict.items()]: list comprehension; desempacota chave e valor do dicionário.
STAGE_COLUMNS = {
    "created": "created_at",
    "approved": "approved_at",
    "shipped": "shipped_at",
    "delivered": "delivered_at",
}


def reduce_batch(events: DataFrame) -> DataFrame:
    """Uma linha por pedido: status do evento de maior sequência + instante de cada etapa do lote."""
    newest_first = Window.partitionBy("order_id").orderBy(F.col("event_sequence").desc(), F.col("event_ts").desc())
    latest = (
        events
        .withColumn("_rank", F.row_number().over(newest_first))
        .where(F.col("_rank") == 1)
        .select("order_id", "status", "event_sequence", "event_ts")
    )
    stages = events.groupBy("order_id").agg(*[
        F.max(F.when(F.col("status") == status, F.col("event_ts"))).alias(column)
        for status, column in STAGE_COLUMNS.items()
    ])
    return latest.join(stages, on="order_id", how="inner")

In [ ]:
# O QUE FAZ: testa reduce_batch com 5 eventos montados à mão, sem stream e sem gravar nada:
#   - pedido A: o delivered (seq 4) chega ANTES do shipped (seq 3) e há um duplicado do created;
#   - pedido B: só o created.
#   Esperado: A fica com status delivered e, mesmo assim, com o instante do shipped preenchido.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(lista_de_tuplas, schema_ddl): DataFrame pequeno a partir de dados Python.
# - datetime(ano, mês, dia, hora, minuto): data/hora do Python.
# - df.collect(): traz as linhas ao driver; aceitável aqui só porque o resultado tem 2 linhas.
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
from datetime import datetime

teste = spark.createDataFrame(
    [
        ("A", "delivered", 4, datetime(2028, 8, 25, 10, 0)),   # chegou primeiro, apesar de ser o último passo
        ("A", "shipped",   3, datetime(2028, 8, 23, 9, 0)),
        ("A", "created",   1, datetime(2028, 8, 22, 8, 0)),
        ("A", "created",   1, datetime(2028, 8, 22, 8, 0)),    # duplicado
        ("B", "created",   1, datetime(2028, 8, 22, 12, 0)),
    ],
    "order_id string, status string, event_sequence int, event_ts timestamp",
)

reduzido = {row["order_id"]: row for row in reduce_batch(teste).collect()}

assert len(reduzido) == 2, "esperado: uma linha por pedido"
assert reduzido["A"]["status"] == "delivered" and reduzido["A"]["event_sequence"] == 4, "A deveria estar delivered (seq 4)"
assert reduzido["A"]["shipped_at"] is not None, "A deveria manter o instante do shipped"
assert reduzido["B"]["status"] == "created" and reduzido["B"]["shipped_at"] is None, "B deveria estar created, sem despacho"

display(reduce_batch(teste).orderBy("order_id"))
print("reduce_batch: OK")

## Teste da leitura em streaming (etapa 5.2)

Roda só com `teste_leitura` diferente de `nao` e **encerra o notebook** antes de qualquer escrita em `gold_orders_live`. Os resultados ficam em `olist.ops.teste_5_2_lotes` (um registro por micro-lote) e o checkpoint de prova em `/Volumes/olist/ops/checkpoints/streaming/teste_5_2/`.

- **`leitura`:** duas leituras em `AvailableNow`, cada uma com checkpoint próprio: `raw` (limpa, sem deduplicação) e `dedup` (a leitura real, com watermark). Cada execução lê **só as linhas novas** da bronze desde a anterior. Confere que `raw` somou exatamente as linhas limpas da bronze e compara `dedup` com as chaves distintas.
- **`robustez`:** numa **cópia** (`SHALLOW CLONE`) da bronze, faz `DELETE` e `INSERT` e confere que a leitura **com** `skipChangeCommits` ignora o `DELETE` e lê só o `INSERT` (sem a opção, o `DELETE` derruba a leitura: `DELTA_SOURCE_TABLE_IGNORE_CHANGES`, observado nas execuções de prova). A bronze real não é tocada.
- Mede, por micro-lote, os segundos entre a gravação do arquivo na landing e a entrada na bronze, e entre o arquivo e a leitura (para comparar com o SLA de 15 min).

In [ ]:
# O QUE FAZ: declara as peças do teste da leitura (tabela de resultados, tratador de micro-lote e
#   executor). Não lê nem grava dados: só declara. A tabela de resultados é criada só quando o teste
#   vai rodar.
#
# FUNÇÕES DECLARADAS:
# - make_stats_handler(name): devolve a função que o foreachBatch chama a cada micro-lote. Ela mede
#   o lote (linhas, pedidos, menor e maior event_ts e os dois tempos de latência) e acrescenta uma
#   linha em ops.teste_5_2_lotes.
# - run_read_test(name, stream_df, checkpoint): roda o stream uma vez com AvailableNow (lê o que
#   chegou desde o checkpoint e termina) e espera o fim.
# - read_totals(name): linhas e lotes já registrados para a leitura "name" (todas as execuções).
#
# FUNÇÕES NATIVAS:
# - df.writeStream.foreachBatch(função): entrega cada micro-lote como DataFrame comum à função.
# - .option("checkpointLocation", pasta): onde o stream guarda até onde já leu.
# - .trigger(availableNow=True): processa tudo o que chegou e termina (único gatilho do serverless).
# - query.awaitTermination(): espera o stream terminar; um erro do stream é lançado aqui.
# - F.unix_timestamp(coluna): segundos desde 1970; a diferença de dois deles são segundos.
# - F.current_timestamp(): horário real do cálculo. F.countDistinct: valores distintos.
# - df.agg(...).select(...): uma linha de totais do micro-lote.
TESTE_LEITURA = dbutils.widgets.get("teste_leitura")

TEST_STATS_TABLE = f"{CATALOG}.ops.teste_5_2_lotes"
TEST_CLONE_TABLE = f"{CATALOG}.ops.teste_5_2_clone"
TEST_CHECKPOINT_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/streaming/teste_5_2"


def make_stats_handler(name: str):
    """Tratador de micro-lote: mede o lote e grava uma linha em ops.teste_5_2_lotes."""
    def handler(batch_df: DataFrame, batch_id: int) -> None:
        (
            batch_df.agg(
                F.count("*").alias("linhas"),
                F.countDistinct("order_id").alias("pedidos"),
                F.min("event_ts").alias("min_event_ts"),
                F.max("event_ts").alias("max_event_ts"),
                F.max(F.unix_timestamp("_ingested_at") - F.unix_timestamp("_source_file_modified_at")).alias("s_arquivo_ate_bronze"),
                F.max(F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("_source_file_modified_at")).alias("s_arquivo_ate_leitura"),
            )
            .select(
                F.lit(name).alias("leitura"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(batch_id).cast("bigint").alias("batch_id"),
                F.current_timestamp().alias("processed_at"),
                "linhas", "pedidos", "min_event_ts", "max_event_ts", "s_arquivo_ate_bronze", "s_arquivo_ate_leitura",
            )
            .write.mode("append").saveAsTable(TEST_STATS_TABLE)
        )
    return handler


def run_read_test(name: str, stream_df: DataFrame, checkpoint: str | None = None) -> None:
    """Uma passada AvailableNow do stream, com o tratador de medição, até terminar."""
    query = (
        stream_df.writeStream
        .foreachBatch(make_stats_handler(name))
        .option("checkpointLocation", checkpoint or f"{TEST_CHECKPOINT_DIR}/{name}")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()


def read_totals(name: str) -> tuple[int, int]:
    """(linhas, lotes) já registrados para a leitura `name`, somando todas as execuções."""
    row = spark.table(TEST_STATS_TABLE).where(F.col("leitura") == name).agg(
        F.coalesce(F.sum("linhas"), F.lit(0)).alias("linhas"), F.count("*").alias("lotes")
    ).first()
    return int(row["linhas"]), int(row["lotes"])


if TESTE_LEITURA != "nao":
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TEST_STATS_TABLE} (
            leitura               STRING    COMMENT 'Leitura de prova: raw (sem deduplicação), dedup (com watermark) ou robustez_*.',
            run_id                STRING    COMMENT 'Execução do notebook.',
            batch_id              BIGINT    COMMENT 'Número do micro-lote no checkpoint da leitura.',
            processed_at          TIMESTAMP COMMENT 'Momento em que o micro-lote foi medido (horário real).',
            linhas                BIGINT    COMMENT 'Linhas do micro-lote (depois da deduplicação, na leitura dedup).',
            pedidos               BIGINT    COMMENT 'Pedidos distintos no micro-lote.',
            min_event_ts          TIMESTAMP COMMENT 'Menor event_ts do micro-lote (tempo simulado).',
            max_event_ts          TIMESTAMP COMMENT 'Maior event_ts do micro-lote (tempo simulado).',
            s_arquivo_ate_bronze  BIGINT    COMMENT 'Maior intervalo, em segundos, entre a gravação do arquivo na landing e a entrada na bronze.',
            s_arquivo_ate_leitura BIGINT    COMMENT 'Maior intervalo, em segundos, entre a gravação do arquivo na landing e a leitura em streaming.'
        )
        COMMENT 'Medições dos testes da leitura em streaming (etapa 5.2): um registro por micro-lote. Pode ser apagada ao fim da fase.'
    """)
print(f"teste_leitura: {TESTE_LEITURA}")

In [ ]:
# O QUE FAZ: teste "leitura" (só roda com teste_leitura = leitura). Lê o feed da bronze duas vezes,
#   cada uma com checkpoint próprio e uma passada AvailableNow, e confere:
#   1. a leitura raw (sem deduplicação) somou EXATAMENTE as linhas limpas da bronze: cada linha foi
#      lida uma vez, e uma execução sem linhas novas lê 0;
#   2. a leitura dedup (a real, com watermark) somou quantas linhas, comparada com as chaves
#      distintas (order_id, event_sequence) da bronze. Diferença é achado para a etapa 5.3, não erro.
#   Mostra os micro-lotes desta execução com os tempos de latência.
#
# FUNÇÕES NATIVAS:
# - df.select(...).distinct().count(): quantidade de combinações distintas de colunas.
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
# - display(df): mostra o resultado em tabela.
if TESTE_LEITURA == "leitura":
    lidas_antes = {nome: read_totals(nome) for nome in ("raw", "dedup")}

    run_read_test("raw", read_events_clean())
    run_read_test("dedup", read_events_stream())

    lidas_depois = {nome: read_totals(nome) for nome in ("raw", "dedup")}
    bronze_limpa = clean_events(spark.table(EVENTS_TABLE))
    total_bronze = bronze_limpa.count()
    chaves_distintas = bronze_limpa.select("order_id", "event_sequence").distinct().count()

    novas_raw = lidas_depois["raw"][0] - lidas_antes["raw"][0]
    novas_dedup = lidas_depois["dedup"][0] - lidas_antes["dedup"][0]
    print(f"raw:   {novas_raw} linhas novas nesta execução em {lidas_depois['raw'][1] - lidas_antes['raw'][1]} micro-lote(s); acumulado {lidas_depois['raw'][0]} de {total_bronze} na bronze")
    print(f"dedup: {novas_dedup} linhas novas nesta execução em {lidas_depois['dedup'][1] - lidas_antes['dedup'][1]} micro-lote(s); acumulado {lidas_depois['dedup'][0]} (chaves distintas na bronze: {chaves_distintas})")

    assert lidas_depois["raw"][0] == total_bronze, f"a leitura raw somou {lidas_depois['raw'][0]} linhas, a bronze tem {total_bronze}"
    print("OK  raw: cada linha limpa da bronze foi lida exatamente uma vez")
    if lidas_depois["dedup"][0] == chaves_distintas:
        print("OK  dedup: soma = chaves distintas da bronze")
    else:
        print(f"ACHADO  dedup: soma {lidas_depois['dedup'][0]} difere das chaves distintas {chaves_distintas} (diferença {lidas_depois['dedup'][0] - chaves_distintas}); investigar na etapa 5.3")

    display(spark.table(TEST_STATS_TABLE).where(F.col("run_id") == RUN_ID).orderBy("leitura", "batch_id"))

In [ ]:
# O QUE FAZ: teste "robustez" (só roda com teste_leitura = robustez): o que a leitura faz quando a
#   tabela de origem sofre UPDATE/DELETE, como acontecerá com a exclusão de cliente (Fase 6).
#   Tudo numa CÓPIA da bronze (SHALLOW CLONE: copia só os metadados, os arquivos de dados são
#   compartilhados), que é apagada no fim. Passos:
#   1. lê a cópia inteira; 2. faz DELETE de um pedido e INSERT de 3 linhas; 3. lê COM skipChangeCommits
#   no mesmo checkpoint (espera só as 3 linhas inseridas). Confere que a bronze real não mudou.
#   O caso SEM skipChangeCommits (a leitura falha com DELTA_SOURCE_TABLE_IGNORE_CHANGES) foi observado
#   em execuções de prova, mas não é afirmado aqui: o Databricks derruba o comando inteiro quando
#   qualquer stream termina com erro, mesmo tratado, e isso falharia o job.
#
# RECURSOS:
# - CREATE OR REPLACE TABLE ... SHALLOW CLONE origem: cópia leve da tabela Delta.
# - DELETE FROM / INSERT INTO ... SELECT: altera a cópia (o DELETE reescreve arquivos: é o "commit de
#   mudança" que a fonte de streaming não aceita por padrão).
# - try / finally: a cópia e o checkpoint de prova são apagados com ou sem erro.
# - dbutils.fs.rm(pasta, recurse=True): apaga a pasta do checkpoint de prova.
if TESTE_LEITURA == "robustez":
    bronze_antes = spark.table(EVENTS_TABLE).count()
    checkpoint = f"{TEST_CHECKPOINT_DIR}/robustez"
    limpa = "_corrupt_record IS NULL AND order_id IS NOT NULL"
    try:
        spark.sql(f"DELETE FROM {TEST_STATS_TABLE} WHERE leitura LIKE 'robustez%'")  # repetível: recomeça as medições
        spark.sql(f"CREATE OR REPLACE TABLE {TEST_CLONE_TABLE} SHALLOW CLONE {EVENTS_TABLE}")
        dbutils.fs.rm(checkpoint, recurse=True)

        run_read_test("robustez_1_leitura_inicial", read_events_clean(TEST_CLONE_TABLE, skip_change_commits=False), checkpoint)
        inicial = read_totals("robustez_1_leitura_inicial")[0]
        print(f"1. leitura inicial da cópia: {inicial} linhas")

        vitima = spark.table(TEST_CLONE_TABLE).where(limpa).select("order_id").first()["order_id"]
        spark.sql(f"DELETE FROM {TEST_CLONE_TABLE} WHERE order_id = '{vitima}'")
        spark.sql(f"INSERT INTO {TEST_CLONE_TABLE} SELECT * FROM {TEST_CLONE_TABLE} WHERE {limpa} LIMIT 3")
        print(f"2. DELETE do pedido {vitima} e INSERT de 3 linhas na cópia")

        run_read_test("robustez_2_com_opcao", read_events_clean(TEST_CLONE_TABLE, skip_change_commits=True), checkpoint)
        novas = read_totals("robustez_2_com_opcao")[0]
        assert novas == 3, f"COM skipChangeCommits esperado 3 linhas (as inseridas); lidas {novas}"
        print(f"3. COM skipChangeCommits: leu {novas} linhas (só o INSERT; o DELETE foi ignorado): OK")
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {TEST_CLONE_TABLE}")
        dbutils.fs.rm(checkpoint, recurse=True)

    assert spark.table(EVENTS_TABLE).count() == bronze_antes, "a bronze real mudou"
    print(f"bronze real intacta ({bronze_antes} linhas); cópia e checkpoint de prova apagados")

In [ ]:
# O QUE FAZ: encerra o notebook depois dos testes da etapa 5.2, antes de qualquer escrita em
#   gold_orders_live (as partes seguintes só rodam com teste_leitura = nao).
#
# FUNÇÃO NATIVA:
# - dbutils.notebook.exit(texto): termina o notebook nesse ponto, com sucesso, devolvendo o texto.
if TESTE_LEITURA != "nao":
    dbutils.notebook.exit(f"Teste '{TESTE_LEITURA}' concluído. Nada foi gravado em {LIVE_TABLE}.")